# UdaPlay Project — Part 1

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# Load environment variables
load_dotenv()
required = ["OPENAI_API_KEY", "OPENAI_BASE_URL", "TAVILY_API_KEY"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"Missing environment variables: {', '.join(missing)}")
print("Environment variables loaded")

Environment variables loaded


### VectorDB Instance

In [4]:
# Instantiate your ChromaDB Client
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [5]:
# Pick one embedding function
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
  api_key = os.environ.get("OPENAI_API_KEY")
)

In [6]:
# Create or get the udaplay collection
if "udaplay" not in [c.name for c in chroma_client.list_collections()]:
   collection = chroma_client.create_collection(   
      name="udaplay",
      embedding_function=embedding_fn
   )
else: 
   collection = chroma_client.get_collection("udaplay")

### Add documents

In [7]:
# Load every game JSON file and index it as one document in the collection
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    # upsert instead of add so re-running this cell is idempotent
    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

print(f"Documents in collection '{collection.name}': {collection.count()}")

Documents in collection 'udaplay': 15


### Semantic search

The collection is now persisted on disk under `chromadb/`. The queries below use wording that does
**not** literally appear in the indexed documents, so any relevant hit is the result of embedding
similarity rather than keyword matching.

In [8]:
def semantic_search(query: str, n_results: int = 3):
    """Query the collection and print the closest games with their similarity distance."""
    results = collection.query(query_texts=[query], n_results=n_results)

    print(f"Query: {query!r}")
    for doc_id, meta, distance in zip(
        results["ids"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        print(
            f"  [{doc_id}] distance={distance:.4f} | "
            f"{meta['Name']} ({meta['YearOfRelease']}, {meta['Platform']}) "
            f"- {meta['Genre']}, published by {meta['Publisher']}"
        )
    print()
    return results


# None of these words appear verbatim in the documents - the matches come from the embeddings
_ = semantic_search("a driving simulation game with lots of licensed cars")
_ = semantic_search("a role-playing game about collecting and training monsters")
_ = semantic_search("game about building and exploring with blocks")
_ = semantic_search("competitive fighting game with Nintendo characters")

Query: 'a driving simulation game with lots of licensed cars'
  [003] distance=0.1366 | Gran Turismo 5 (2010, PlayStation 3) - Racing, published by Sony Computer Entertainment
  [001] distance=0.1413 | Gran Turismo (1997, PlayStation 1) - Racing, published by Sony Computer Entertainment
  [002] distance=0.1958 | Grand Theft Auto: San Andreas (2004, PlayStation 2) - Action-adventure, published by Rockstar Games



Query: 'a role-playing game about collecting and training monsters'
  [006] distance=0.1971 | Pokémon Gold and Silver (1999, Game Boy Color) - Role-playing, published by Nintendo
  [007] distance=0.1990 | Pokémon Ruby and Sapphire (2002, Game Boy Advance) - Role-playing, published by Nintendo
  [008] distance=0.2150 | Super Mario World (1990, Super Nintendo Entertainment System (SNES)) - Platformer, published by Nintendo



Query: 'game about building and exploring with blocks'
  [014] distance=0.1317 | Minecraft (2014, Xbox One) - Sandbox, Survival, published by Mojang Studios
  [009] distance=0.2089 | Super Mario 64 (1996, Nintendo 64) - Platformer, published by Nintendo
  [008] distance=0.2111 | Super Mario World (1990, Super Nintendo Entertainment System (SNES)) - Platformer, published by Nintendo



Query: 'competitive fighting game with Nintendo characters'
  [010] distance=0.1415 | Super Smash Bros. Melee (2001, GameCube) - Fighting, published by Nintendo
  [012] distance=0.1846 | Mario Kart 8 Deluxe (2017, Nintendo Switch) - Racing, published by Nintendo
  [009] distance=0.2021 | Super Mario 64 (1996, Nintendo 64) - Platformer, published by Nintendo

